# Building AI Agents with OpenAI (Lesson 17 Assignment & Frameworks)

In [1]:
import os 
import json 
import pandas as pd 
from openai import OpenAI

client=OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL="gpt-4o-mini"

## Part 1: Defining Tools for Our Agent

In [3]:
def get_startup_metrics(department: str) -> str:
    """Retrieves key performance metrics for a specific department in the education startup."""
    metrics = {
        "finance": "Current runway: 14 months. Burn rate: $45k/month. CAC: $120.",
        "product": "Active MAU: 12,000. Course completion rate: 64%.",
        "marketing": "Website traffic: 45k visits/mo. Conversion rate: 3.2%."
    }

    return metrics.get(department.lower(),"Department metrics not found.")

tools_registry={
    "get_startup_metrics":get_startup_metrics
}

tools_definition = [
    {
        "type": "function",
        "function": {
            "name": "get_startup_metrics",
            "description": "Get key performance metrics for a startup department.",
            "parameters": {
                "type": "object",
                "properties": {
                    "department": {
                        "type": "string",
                        "description": "Department name: finance, product, or marketing.",
                    }
                },
                "required": ["department"],
                
            }
        }
    }
]


## Part 2: Implementing the Agent State & Execution Loop

In [5]:
def run_agent_with_tools(user_query):
    """Executes an agent turn supporting tool calls and maintaining conversation state."""
    messages = [
        {"role": "system", "content": "You are an intelligent startup AI assistant with access to company metrics tools."},
        {"role": "user", "content": user_query}
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools_definition,
        tool_choice="auto"
    )

    response_message= response.choices[0].message
    messages.append(response_message)

    if response_message.tool_calls:
        for tool_call in response_message.tool_calls:
            function_name=tool_call.function.name
            function_args=json.loads(tool_call.function.arguments)

            print(f"[Agent Action] Calling tool'{function_name}' with args{function_args}...")

            tool_function=tools_registry[function_name]
            tool_output=tool_function(**function_args)

            messages.append({
                "role":"tool",
                "tool_call_id":tool_call.id,
                "content":str(tool_output)
            })

        final_response=client.chat.completions.create(
            model=MODEL,
            messages=messages
        )

        return final_response.choices[0].message.content

    return response_message.content

print(run_agent_with_tools("Can you check our finance department metrics?"))    



[Agent Action] Calling tool'get_startup_metrics' with args{'department': 'finance'}...
Here are the current metrics for the finance department:

- **Runway:** 14 months
- **Burn Rate:** $45,000 per month
- **Customer Acquisition Cost (CAC):** $120

If you need further analysis or insights based on these metrics, feel free to ask!


## Part 3: Assignment - Education Startup Business Meeting Simulation

In [6]:
def simulate_department_meeting(product_pitch):
    """Simulates a multidisciplinary startup meeting where different department personas review a pitch."""
    departments = [
        {
            "role": "Finance Department",
            "persona": "You are the CFO. Focus heavily on budget, ROI, customer acquisition cost (CAC), and financial runway."
        },
        {
            "role": "Product Department",
            "persona": "You are the Head of Product. Focus on user experience, feature complexity, adoption friction, and tech stack."
        },
        {
            "role": "Marketing Department",
            "persona": "You are the CMO. Focus on target audience positioning, go-to-market strategy, and brand alignment."
        }
    ]
    
    meeting_results = []
    
    for dept in departments:
        print(f"\n--- Consulting {dept['role']} ---")
        messages = [
            {"role": "system", "content": dept['persona']},
            {"role": "user", "content": f"Here is the new product pitch for our education startup:\n\n'{product_pitch}'\n\nProvide your critical evaluation and ask 2 sharp follow-up questions to refine the idea."}
        ]
        
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.4,
            max_tokens=250
        )
        
        feedback = response.choices[0].message.content
        print(feedback)
        meeting_results.append({"Department": dept['role'], "Feedback": feedback})
        
    return pd.DataFrame(meeting_results)

# Run the business meeting simulation with a sample pitch
pitch = "An AI-powered personalized tutor chatbot embedded in WhatsApp that helps high school students learn calculus through gamified quizzes."
meeting_df = simulate_department_meeting(pitch)


--- Consulting Finance Department ---
**Critical Evaluation:**

1. **Market Demand and Customer Acquisition Cost (CAC):** The education technology market is growing, particularly in personalized learning solutions. However, we need to validate the demand specifically for a calculus tutor chatbot. Understanding the target market's willingness to pay and the channels through which we can effectively reach high school students is crucial. We should conduct market research to estimate the CAC and ensure it aligns with our budget and financial runway.

2. **Budget and ROI:** Developing an AI-powered chatbot will require significant upfront investment in technology, content creation, and possibly partnerships with educational institutions. We need to outline a detailed budget that includes development costs, marketing expenses, and ongoing operational costs. Additionally, we should project the expected ROI based on pricing models, subscription rates, and potential scale. 

3. **Competitive 